# Introduction

### Evaluating the Faithfulness of LLM-Generated Explanations for Fact Verification

**Author:** Ilnaza Saifutdinova
**Programme:** B.Sc. Software Engineering — University of Applied Sciences Europe
**Dataset:** FEVER Shared Task Dev Set (19,998 labelled claims)

### Overview
This notebook implements the experimental pipeline for evaluating whether
LLM-generated explanations for fact verification are faithful to the gold
evidence provided in the FEVER benchmark.

**Three prompting conditions:**
- Condition 1: Answer only (baseline)
- Condition 2: Answer + free explanation
- Condition 3: Gold evidence + answer + explanation (RAG)

## Setup and Imports

In [14]:
import json
import pathlib
import time

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from rouge_score import rouge_scorer
import seaborn as sns
import plotly.express as px

# Set seaborn style globally — applies to all matplotlib plots too
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.2)

## 1. Load FEVER Dataset

In [15]:
DATA_PATH = pathlib.Path("data/shared_task_dev.jsonl")

df = pd.read_json(DATA_PATH, lines=True)

print("Dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
df.head()

Dataset shape: (19998, 5)

Column names: ['id', 'verifiable', 'label', 'claim', 'evidence']


,id,verifiable,label,claim,evidence
0,91198,NOT VERIFIABLE,NOT ENOUGH INFO,Colin Kaepernick became a starting quarterback...,"[[[108548, None, None, None]]]"
1,194462,NOT VERIFIABLE,NOT ENOUGH INFO,Tilda Swinton is a vegan.,"[[[227768, None, None, None]]]"
2,137334,VERIFIABLE,SUPPORTS,Fox 2000 Pictures released the film Soul Food.,"[[[289914, 283015, Soul_Food_-LRB-film-RRB-, 0..."
3,166626,NOT VERIFIABLE,NOT ENOUGH INFO,Anne Rice was born in New Jersey.,"[[[191656, None, None, None], [191657, None, N..."
4,111897,VERIFIABLE,REFUTES,Telemundo is a English-language television net...,"[[[131371, 146144, Telemundo, 0]], [[131371, 1..."


## 2. Data Preprocessing

In [16]:
# Label distribution
print("Label distribution:")
print(df["label"].value_counts())
print()

# What does evidence look like for one claim?
print("Evidence structure (SUPPORTS example):")
supports_ex = df[df["label"] == "SUPPORTS"].iloc[0]
print(f"Claim: {supports_ex['claim']}")
print(f"Evidence: {supports_ex['evidence']}")

Label distribution:
label
NOT ENOUGH INFO    6666
SUPPORTS           6666
REFUTES            6666
Name: count, dtype: int64

Evidence structure (SUPPORTS example):
Claim: Fox 2000 Pictures released the film Soul Food.
Evidence: [[[289914, 283015, 'Soul_Food_-LRB-film-RRB-', 0]], [[291259, 284217, 'Soul_Food_-LRB-film-RRB-', 0]], [[293412, 285960, 'Soul_Food_-LRB-film-RRB-', 0]], [[337212, 322620, 'Soul_Food_-LRB-film-RRB-', 0]], [[337214, 322622, 'Soul_Food_-LRB-film-RRB-', 0]]]


### 2.1 Parse Evidence Structure
Flatten the nested evidence pointers and extract readable Wikipedia page names and sentence counts per claim.

In [17]:
def parse_evidence(evidence):
    """Extract unique (page, sentence_id) pairs from evidence sets."""
    seen = set()
    for ev_set in evidence:
        for ev in ev_set:
            if ev[2] is not None:  # ev[2] = wikipedia page
                seen.add((ev[2], ev[3]))  # (page, sentence_id)
    return list(seen)

def clean_page_name(page):
    """Convert Wikipedia URL format to readable name."""
    return page.replace("_", " ").replace("-LRB-", "(").replace("-RRB-", ")")

df["evidence_parsed"] = df["evidence"].apply(parse_evidence)
df["evidence_pages"] = df["evidence_parsed"].apply(
    lambda evs: [clean_page_name(e[0]) for e in evs]
)
df["n_evidence"] = df["evidence_parsed"].apply(len)

print(df[["claim", "label", "evidence_pages", "n_evidence"]].head(5))

                                               claim            label  \
0  Colin Kaepernick became a starting quarterback...  NOT ENOUGH INFO   
1                          Tilda Swinton is a vegan.  NOT ENOUGH INFO   
2     Fox 2000 Pictures released the film Soul Food.         SUPPORTS   
3                  Anne Rice was born in New Jersey.  NOT ENOUGH INFO   
4  Telemundo is a English-language television net...          REFUTES   

                                      evidence_pages  n_evidence  
0                                                 []           0  
1                                                 []           0  
2                                 [Soul Food (film)]           1  
3                                                 []           0  
4  [Hispanic and Latino Americans, Telemundo, Tel...           5  


### 2.2 Normalize Labels
Standardize label formatting and verify class distribution.

In [18]:
# Normalize label formatting (FEVER uses caps, keep consistent)
df["label"] = df["label"].str.strip().str.upper()

# Verify
print("Normalized label counts:")
print(df["label"].value_counts())
print(f"\nAny nulls: {df.isnull().sum().to_dict()}")
print(f"\nDataset shape after preprocessing: {df.shape}")

Normalized label counts:
label
NOT ENOUGH INFO    6666
SUPPORTS           6666
REFUTES            6666
Name: count, dtype: int64

Any nulls: {'id': 0, 'verifiable': 0, 'label': 0, 'claim': 0, 'evidence': 0, 'evidence_parsed': 0, 'evidence_pages': 0, 'n_evidence': 0}

Dataset shape after preprocessing: (19998, 8)


### 2.3 Filter for Stratified Sampling
For the experiments we use claims where evidence is clearly defined:
- SUPPORTS and REFUTES: exactly 1 evidence sentence (cleanest for faithfulness evaluation)
- NOT ENOUGH INFO: no evidence by definition

From 19,998 total claims:
- 4,868 SUPPORTS with single evidence sentence
- 4,898 REFUTES with single evidence sentence
- 6,666 NOT ENOUGH INFO
- 100 sampled per class → 300 claims total (SEED = 2026)

data/
├── shared_task_dev.jsonl    ← full dataset (19,998 claims)
└── sample_300.jsonl         ← your experiment sample (300 claims)

In [19]:
SEED = 2026
N_PER_CLASS = 100

# SUPPORTS and REFUTES: only single-sentence evidence (cleanest for faithfulness eval)
df_sup = df[(df["label"] == "SUPPORTS") & (df["n_evidence"] == 1)]
df_ref = df[(df["label"] == "REFUTES")  & (df["n_evidence"] == 1)]
df_nei = df[df["label"] == "NOT ENOUGH INFO"]

print(f"Available SUPPORTS (1 evidence): {len(df_sup)}")
print(f"Available REFUTES  (1 evidence): {len(df_ref)}")
print(f"Available NEI:                   {len(df_nei)}")

# Stratified sample
sample = pd.concat([
    df_sup.sample(n=N_PER_CLASS, random_state=SEED),
    df_ref.sample(n=N_PER_CLASS, random_state=SEED),
    df_nei.sample(n=N_PER_CLASS, random_state=SEED),
]).reset_index(drop=True)

print(f"\nFinal sample: {len(sample)} claims")
print(sample["label"].value_counts())

sample.to_json("data/sample_300.jsonl", orient="records", lines=True)
print("\nSaved to data/sample_300.jsonl")

Available SUPPORTS (1 evidence): 4868
Available REFUTES  (1 evidence): 4898
Available NEI:                   6666

Final sample: 300 claims
label
SUPPORTS           100
REFUTES            100
NOT ENOUGH INFO    100
Name: count, dtype: int64

Saved to data/sample_300.jsonl


> **Note:** 73% of SUPPORTS (4,868/6,666) and 74% of REFUTES (4,898/6,666)
> claims have single-sentence evidence, confirming the filtered sample
> remains representative of the full dataset.

> **Note:** Also those numbers tell you something useful — roughly 73% of SUPPORTS and 74% of REFUTES claims
> have single-sentence evidence. So filtering to single-sentence didn't throw away much, which means your
> sample is still representative of the dataset overall. Worth one sentence in your methodology chapter.

## 3. Exploratory Data Analysis

Before running any LLM experiments, we visualise the structure of the
sampled dataset to confirm balance and understand claim/evidence characteristics.

In [20]:
pathlib.Path("figures").mkdir(exist_ok=True)

### 3.1 Label Distribution

In [40]:
label_counts = sample["label"].value_counts().reset_index()
label_counts.columns = ["label", "count"]

fig = px.bar(
    label_counts,
    x="label",
    y="count",
    color="label",
    color_discrete_map={
        "SUPPORTS": "#2ecc71",
        "REFUTES": "#e74c3c",
        "NOT ENOUGH INFO": "#95a5a6"
    },
    title="Sample label distribution (n=300)",
    labels={"label": "Label", "count": "Number of claims"},
    text_auto=True
)
fig.update_layout(showlegend=False, plot_bgcolor="white")
fig.show()

### 3.2 Claim Length Distribution
How many words are in each claim, broken down by label.

In [41]:
sample["claim_length"] = sample["claim"].str.split().str.len()

fig = px.histogram(
    sample,
    x="claim_length",
    color="label",
    color_discrete_map={
        "SUPPORTS": "#2ecc71",
        "REFUTES": "#e74c3c",
        "NOT ENOUGH INFO": "#95a5a6"
    },
    barmode="overlay",
    opacity=0.7,
    nbins=30,
    title="Claim length distribution by label",
    labels={"claim_length": "Number of words", "count": "Number of claims"}
)
fig.update_layout(plot_bgcolor="white", legend_title="Label")
fig.show()

>This overlapping histogram will show whether REFUTES claims tend to be longer or shorter than SUPPORTS —
> which is actually an interesting finding for your thesis if there's a difference

> **Note:** All three label classes show similar claim length distributions
> (peak 7–9 words, right-skewed), suggesting claim length is not a
> confounding factor in the experiments.

### 3.3 Evidence Count Distribution
Number of evidence sentences per claim (SUPPORTS and REFUTES only —
NEI has no evidence by definition).

In [42]:
df_ev = sample[sample["label"].isin(["SUPPORTS", "REFUTES"])].copy()

fig = px.histogram(
    df_ev,
    x="n_evidence",
    color="label",
    color_discrete_map={
        "SUPPORTS": "#2ecc71",
        "REFUTES": "#e74c3c",
    },
    barmode="group",
    title="Evidence sentence count (SUPPORTS vs REFUTES)",
    labels={"n_evidence": "Number of evidence sentences", "count": "Number of claims"},
    nbins=10
)
fig.update_layout(plot_bgcolor="white", legend_title="Label")
fig.show()

> **Note:** All sampled SUPPORTS and REFUTES claims have exactly 1 gold
> evidence sentence, ensuring a clean and consistent input for the
> RAG prompting condition (Condition 3).